# Blue Archive ガチャシミュレーション

旧仕様・新仕様を切り替え可能なモンテカルロシミュレーションです。

主な特徴:

- 星3ピックアップ確率、別バナーPU確率、星3全体確率を設定可能
- 新仕様では100チャージで星3確定・50%で対象PU
- 新仕様では200チャージで対象PU確定
- 対象PUを引いた瞬間にチャージを0へリセット
- `initial_charge` により任意のチャージ数から開始可能
- 終了条件を関数として自由に差し替え可能
- 集計関数を追加しやすい構成
- 乱数seedを固定して再現可能


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from enum import Enum, auto
from typing import Callable, Iterable, Sequence
import math
import random
import statistics

import matplotlib.pyplot as plt


## 1. データ構造と設定

In [ ]:
class PullType(Enum):
    """1回の募集結果の分類。"""

    STAR1 = auto()
    STAR2 = auto()
    STAR3_OTHER = auto()
    STAR3_OFFPICKUP = auto()
    STAR3_PICKUP = auto()


@dataclass(frozen=True)
class GachaConfig:
    """ガチャ仕様と確率をまとめた設定。

    確率は小数で指定する。
    例:
        0.007 = 0.7%
        0.03  = 3.0%
    """

    pickup_rate: float = 0.007
    offpickup_rate: float = 0.00022115
    star3_rate: float = 0.03

    # 星2以上の合計確率。
    # 星2確率 = star2_or_higher_rate - star3_rate
    star2_or_higher_rate: float = 0.215

    # True: 呼び出しチャージ仕様を使用
    # False: 通常抽選のみ
    use_new_system: bool = True

    # シミュレーション開始時点のチャージ数
    initial_charge: int = 0

    # 100チャージ時に対象PUとなる確率
    charge_100_pickup_rate: float = 0.5

    # チャージ保証が発生する閾値
    charge_star3_threshold: int = 100
    charge_pickup_threshold: int = 200

    def __post_init__(self) -> None:
        probabilities = {
            "pickup_rate": self.pickup_rate,
            "offpickup_rate": self.offpickup_rate,
            "star3_rate": self.star3_rate,
            "star2_or_higher_rate": self.star2_or_higher_rate,
            "charge_100_pickup_rate": self.charge_100_pickup_rate,
        }

        for name, value in probabilities.items():
            if not 0.0 <= value <= 1.0:
                raise ValueError(f"{name} must be between 0 and 1: {value}")

        if self.pickup_rate + self.offpickup_rate > self.star3_rate:
            raise ValueError(
                "pickup_rate + offpickup_rate must not exceed star3_rate."
            )

        if self.star3_rate > self.star2_or_higher_rate:
            raise ValueError(
                "star3_rate must not exceed star2_or_higher_rate."
            )

        if self.charge_star3_threshold <= 0:
            raise ValueError("charge_star3_threshold must be positive.")

        if self.charge_pickup_threshold <= self.charge_star3_threshold:
            raise ValueError(
                "charge_pickup_threshold must exceed charge_star3_threshold."
            )

        if not 0 <= self.initial_charge < self.charge_pickup_threshold:
            raise ValueError(
                "initial_charge must satisfy "
                f"0 <= initial_charge < {self.charge_pickup_threshold}."
            )

    @property
    def other_star3_rate(self) -> float:
        """対象PUと別バナーPUを除く星3確率。"""
        return self.star3_rate - self.pickup_rate - self.offpickup_rate

    @property
    def star2_rate(self) -> float:
        """星2確率。"""
        return self.star2_or_higher_rate - self.star3_rate

    @property
    def star1_rate(self) -> float:
        """星1確率。"""
        return 1.0 - self.star2_or_higher_rate


@dataclass(frozen=True)
class PullResult:
    """1回の募集結果。"""

    pull_index: int
    pull_type: PullType
    by_charge: bool = False
    charge_before_pull: int = 0
    charge_after_pull: int = 0

    @property
    def is_star3(self) -> bool:
        return self.pull_type in {
            PullType.STAR3_OTHER,
            PullType.STAR3_OFFPICKUP,
            PullType.STAR3_PICKUP,
        }

    @property
    def is_pickup(self) -> bool:
        return self.pull_type is PullType.STAR3_PICKUP

    @property
    def is_offpickup(self) -> bool:
        return self.pull_type is PullType.STAR3_OFFPICKUP


@dataclass
class SimulationState:
    """1試行中の状態。"""

    pull_count: int = 0
    charge: int = 0
    pickup_count: int = 0
    offpickup_count: int = 0
    star3_count: int = 0
    history: list[PullResult] = field(default_factory=list)

    @property
    def has_pickup(self) -> bool:
        return self.pickup_count >= 1


@dataclass(frozen=True)
class SimulationResult:
    """1試行終了時の結果。"""

    pull_count: int
    final_charge: int
    pickup_count: int
    offpickup_count: int
    star3_count: int
    history: tuple[PullResult, ...]

    @property
    def has_pickup(self) -> bool:
        return self.pickup_count >= 1

    @classmethod
    def from_state(cls, state: SimulationState) -> "SimulationResult":
        return cls(
            pull_count=state.pull_count,
            final_charge=state.charge,
            pickup_count=state.pickup_count,
            offpickup_count=state.offpickup_count,
            star3_count=state.star3_count,
            history=tuple(state.history),
        )


## 2. ガチャエンジン

In [ ]:
class GachaEngine:
    """ガチャ抽選ロジック。

    `pull` は状態を1回分だけ更新し、PullResultを返す。
    """

    def __init__(
        self,
        config: GachaConfig,
        rng: random.Random | None = None,
    ) -> None:
        self.config = config
        self.rng = rng if rng is not None else random.Random()

    def _draw_normal(self) -> PullType:
        """通常抽選を1回行う。"""
        x = self.rng.random()

        if x < self.config.pickup_rate:
            return PullType.STAR3_PICKUP

        if x < self.config.pickup_rate + self.config.offpickup_rate:
            return PullType.STAR3_OFFPICKUP

        if x < self.config.star3_rate:
            return PullType.STAR3_OTHER

        if x < self.config.star2_or_higher_rate:
            return PullType.STAR2

        return PullType.STAR1

    def _draw_charge_guarantee(
        self,
        charge_before_pull: int,
    ) -> tuple[PullType, bool]:
        """新仕様のチャージ保証を判定する。

        戻り値:
            (pull_type, by_charge)

        例:
            charge_before_pull == 99
            -> 次の1回が100チャージ目

            charge_before_pull == 199
            -> 次の1回が200チャージ目
        """
        if not self.config.use_new_system:
            return self._draw_normal(), False

        next_charge = charge_before_pull + 1

        if next_charge == self.config.charge_pickup_threshold:
            return PullType.STAR3_PICKUP, True

        if next_charge == self.config.charge_star3_threshold:
            if self.rng.random() < self.config.charge_100_pickup_rate:
                return PullType.STAR3_PICKUP, True
            return PullType.STAR3_OFFPICKUP, True

        return self._draw_normal(), False

    def pull(self, state: SimulationState) -> PullResult:
        """1回引き、状態を更新する。"""
        charge_before = state.charge
        pull_type, by_charge = self._draw_charge_guarantee(charge_before)

        state.pull_count += 1
        state.charge += 1

        if pull_type in {
            PullType.STAR3_OTHER,
            PullType.STAR3_OFFPICKUP,
            PullType.STAR3_PICKUP,
        }:
            state.star3_count += 1

        if pull_type is PullType.STAR3_OFFPICKUP:
            state.offpickup_count += 1

        if pull_type is PullType.STAR3_PICKUP:
            state.pickup_count += 1

            # 新仕様では対象PUを引いた瞬間にチャージを0へ戻す。
            if self.config.use_new_system:
                state.charge = 0

        result = PullResult(
            pull_index=state.pull_count,
            pull_type=pull_type,
            by_charge=by_charge,
            charge_before_pull=charge_before,
            charge_after_pull=state.charge,
        )

        state.history.append(result)
        return result


## 3. 終了条件

In [ ]:
StopCondition = Callable[[SimulationState], bool]


def stop_at_pull_count(max_pulls: int) -> StopCondition:
    """指定回数に達したら終了する終了条件を返す。"""
    if max_pulls <= 0:
        raise ValueError("max_pulls must be positive.")

    def condition(state: SimulationState) -> bool:
        return state.pull_count >= max_pulls

    return condition


def stop_on_pickup(state: SimulationState) -> bool:
    """対象PUを1枚以上引いたら終了。"""
    return state.pickup_count >= 1


def stop_on_pickup_count(target_count: int) -> StopCondition:
    """対象PUを指定枚数引いたら終了。"""
    if target_count <= 0:
        raise ValueError("target_count must be positive.")

    def condition(state: SimulationState) -> bool:
        return state.pickup_count >= target_count

    return condition


def any_condition(*conditions: StopCondition) -> StopCondition:
    """いずれかの条件を満たしたら終了。"""
    if not conditions:
        raise ValueError("At least one condition is required.")

    def condition(state: SimulationState) -> bool:
        return any(fn(state) for fn in conditions)

    return condition


def all_conditions(*conditions: StopCondition) -> StopCondition:
    """すべての条件を満たしたら終了。"""
    if not conditions:
        raise ValueError("At least one condition is required.")

    def condition(state: SimulationState) -> bool:
        return all(fn(state) for fn in conditions)

    return condition


# 使用例:
stop_pickup_or_200 = any_condition(
    stop_on_pickup,
    stop_at_pull_count(200),
)


## 4. シミュレーション実行

In [ ]:
def simulate_once(
    engine: GachaEngine,
    stop_condition: StopCondition,
    *,
    safety_max_pulls: int = 100_000,
) -> SimulationResult:
    """1試行を実行する。

    safety_max_pulls は終了条件の設定ミスによる無限ループを防ぐ。
    """
    state = SimulationState(
        charge=engine.config.initial_charge,
    )

    while not stop_condition(state):
        if state.pull_count >= safety_max_pulls:
            raise RuntimeError(
                "safety_max_pulls was reached. "
                "Check the stop condition."
            )
        engine.pull(state)

    return SimulationResult.from_state(state)


def simulate_many(
    config: GachaConfig,
    n_simulations: int,
    stop_condition: StopCondition,
    *,
    seed: int | None = 42,
    safety_max_pulls: int = 100_000,
) -> list[SimulationResult]:
    """複数回のモンテカルロシミュレーションを実行する。"""
    if n_simulations <= 0:
        raise ValueError("n_simulations must be positive.")

    rng = random.Random(seed)
    engine = GachaEngine(config=config, rng=rng)

    return [
        simulate_once(
            engine=engine,
            stop_condition=stop_condition,
            safety_max_pulls=safety_max_pulls,
        )
        for _ in range(n_simulations)
    ]


## 5. 統計量

In [ ]:
StatisticFunction = Callable[[Sequence[SimulationResult]], tuple[str, float]]


def _require_results(
    results: Sequence[SimulationResult],
) -> None:
    if not results:
        raise ValueError("results must not be empty.")


def stat_pickup_acquisition_rate(
    results: Sequence[SimulationResult],
) -> tuple[str, float]:
    _require_results(results)
    value = sum(result.has_pickup for result in results) / len(results)
    return "PU入手率", value


def stat_mean_star3_count(
    results: Sequence[SimulationResult],
) -> tuple[str, float]:
    _require_results(results)
    return "平均★3枚数", statistics.fmean(
        result.star3_count for result in results
    )


def stat_mean_pickup_count(
    results: Sequence[SimulationResult],
) -> tuple[str, float]:
    _require_results(results)
    return "平均PU枚数", statistics.fmean(
        result.pickup_count for result in results
    )


def stat_mean_pull_count(
    results: Sequence[SimulationResult],
) -> tuple[str, float]:
    _require_results(results)
    return "平均連数", statistics.fmean(
        result.pull_count for result in results
    )


def stat_median_pull_count(
    results: Sequence[SimulationResult],
) -> tuple[str, float]:
    _require_results(results)
    return "連数中央値", float(statistics.median(
        result.pull_count for result in results
    ))


def make_pull_percentile_stat(
    percentile: float,
) -> StatisticFunction:
    """連数分布の任意パーセンタイル統計量を作る。"""
    if not 0.0 <= percentile <= 1.0:
        raise ValueError("percentile must be between 0 and 1.")

    def statistic(
        results: Sequence[SimulationResult],
    ) -> tuple[str, float]:
        _require_results(results)
        values = sorted(result.pull_count for result in results)

        # 線形補間によるパーセンタイル
        position = percentile * (len(values) - 1)
        lower_index = math.floor(position)
        upper_index = math.ceil(position)

        if lower_index == upper_index:
            value = float(values[lower_index])
        else:
            weight = position - lower_index
            value = (
                values[lower_index] * (1.0 - weight)
                + values[upper_index] * weight
            )

        return f"連数{percentile:.0%}点", value

    return statistic


def stat_reach_200_rate(
    results: Sequence[SimulationResult],
) -> tuple[str, float]:
    _require_results(results)
    value = sum(
        result.pull_count >= 200
        for result in results
    ) / len(results)
    return "200連到達率", value


def stat_pickup_two_or_more_rate(
    results: Sequence[SimulationResult],
) -> tuple[str, float]:
    _require_results(results)
    value = sum(
        result.pickup_count >= 2
        for result in results
    ) / len(results)
    return "PU2枚以上の確率", value


def stat_zero_star3_rate(
    results: Sequence[SimulationResult],
) -> tuple[str, float]:
    _require_results(results)
    value = sum(
        result.star3_count == 0
        for result in results
    ) / len(results)
    return "★3が0枚の確率", value


DEFAULT_STATISTICS: list[StatisticFunction] = [
    stat_pickup_acquisition_rate,
    stat_mean_star3_count,
    stat_mean_pickup_count,
    stat_mean_pull_count,
    stat_median_pull_count,
    make_pull_percentile_stat(0.90),
    make_pull_percentile_stat(0.95),
    stat_reach_200_rate,
    stat_pickup_two_or_more_rate,
    stat_zero_star3_rate,
]


def summarize_results(
    results: Sequence[SimulationResult],
    statistics_functions: Iterable[StatisticFunction] = DEFAULT_STATISTICS,
) -> dict[str, float]:
    """指定した統計量関数を順番に実行する。"""
    summary: dict[str, float] = {}

    for function in statistics_functions:
        name, value = function(results)
        summary[name] = value

    return summary


def print_summary(summary: dict[str, float]) -> None:
    """統計量を見やすく表示する。"""
    probability_keywords = ("率", "確率")

    for name, value in summary.items():
        if any(keyword in name for keyword in probability_keywords):
            print(f"{name}: {value:.4%}")
        else:
            print(f"{name}: {value:.4f}")


## 6. 可視化

In [ ]:
def plot_pull_histogram(
    results: Sequence[SimulationResult],
    *,
    bins: int = 40,
) -> None:
    """終了までの連数のヒストグラム。"""
    _require_results(results)

    values = [result.pull_count for result in results]

    plt.figure(figsize=(9, 5))
    plt.hist(values, bins=bins)
    plt.xlabel("Pull count")
    plt.ylabel("Frequency")
    plt.title("Distribution of pull counts")
    plt.tight_layout()
    plt.show()


def plot_pull_cdf(
    results: Sequence[SimulationResult],
) -> None:
    """終了までの連数の経験累積分布関数。"""
    _require_results(results)

    values = sorted(result.pull_count for result in results)
    cumulative = [
        (index + 1) / len(values)
        for index in range(len(values))
    ]

    plt.figure(figsize=(9, 5))
    plt.step(values, cumulative, where="post")
    plt.xlabel("Pull count")
    plt.ylabel("Cumulative probability")
    plt.title("Empirical CDF of pull counts")
    plt.ylim(0.0, 1.0)
    plt.tight_layout()
    plt.show()


def plot_star3_histogram(
    results: Sequence[SimulationResult],
) -> None:
    """星3枚数のヒストグラム。"""
    _require_results(results)

    values = [result.star3_count for result in results]
    min_value = min(values)
    max_value = max(values)

    bins = [
        value - 0.5
        for value in range(min_value, max_value + 2)
    ]

    plt.figure(figsize=(9, 5))
    plt.hist(values, bins=bins)
    plt.xlabel("Number of ★3")
    plt.ylabel("Frequency")
    plt.title("Distribution of ★3 counts")
    plt.tight_layout()
    plt.show()


## 7. 動作確認用の簡易テスト

In [ ]:
def run_sanity_checks() -> None:
    """チャージ閾値とリセット仕様の簡易テスト。"""

    # 199チャージ開始なら次の1回はPU確定。
    config_199 = GachaConfig(
        use_new_system=True,
        initial_charge=199,
    )
    engine_199 = GachaEngine(config_199, random.Random(1))
    state_199 = SimulationState(charge=config_199.initial_charge)
    result_200 = engine_199.pull(state_199)

    assert result_200.by_charge is True
    assert result_200.pull_type is PullType.STAR3_PICKUP
    assert state_199.charge == 0
    assert state_199.pickup_count == 1

    # 99チャージ開始なら次の1回は星3確定。
    config_99 = GachaConfig(
        use_new_system=True,
        initial_charge=99,
    )
    engine_99 = GachaEngine(config_99, random.Random(1))
    state_99 = SimulationState(charge=config_99.initial_charge)
    result_100 = engine_99.pull(state_99)

    assert result_100.by_charge is True
    assert result_100.is_star3 is True

    # 通常抽選でPUを引いてもチャージは0へ戻る。
    config_forced_pickup = GachaConfig(
        pickup_rate=1.0,
        offpickup_rate=0.0,
        star3_rate=1.0,
        star2_or_higher_rate=1.0,
        use_new_system=True,
        initial_charge=50,
    )
    engine_forced = GachaEngine(
        config_forced_pickup,
        random.Random(1),
    )
    state_forced = SimulationState(
        charge=config_forced_pickup.initial_charge,
    )
    forced_result = engine_forced.pull(state_forced)

    assert forced_result.pull_type is PullType.STAR3_PICKUP
    assert state_forced.charge == 0

    print("All sanity checks passed.")


run_sanity_checks()


## 8. 実行例

In [ ]:
# --- 設定値 ---

config = GachaConfig(
    pickup_rate=0.007,          # 対象PU: 0.7%
    offpickup_rate=0.00022115,  # 別バナーPU: 例として約0.022115%
    star3_rate=0.03,            # 星3全体: 3.0%
    star2_or_higher_rate=0.215,
    use_new_system=True,
    initial_charge=0,           # 任意の開始チャージ数
    charge_100_pickup_rate=0.5,
    charge_star3_threshold=100,
    charge_pickup_threshold=200,
)

N_SIMULATIONS = 100_000
SEED = 42

# 例: PUを引くか、200連に到達したら終了
stop_condition = any_condition(
    stop_on_pickup,
    stop_at_pull_count(200),
)

results = simulate_many(
    config=config,
    n_simulations=N_SIMULATIONS,
    stop_condition=stop_condition,
    seed=SEED,
)

summary = summarize_results(results)
print_summary(summary)


In [ ]:
plot_pull_histogram(results)
plot_pull_cdf(results)
plot_star3_histogram(results)


## 9. 終了条件の変更例

```python
# 200連固定
stop_condition = stop_at_pull_count(200)

# PUを引くまで
stop_condition = stop_on_pickup

# PUを2枚引くか400連で終了
stop_condition = any_condition(
    stop_on_pickup_count(2),
    stop_at_pull_count(400),
)
```

## 10. 統計量の追加例

```python
def stat_mean_final_charge(results):
    return (
        "平均終了時チャージ",
        statistics.fmean(r.final_charge for r in results),
    )

custom_statistics = [
    *DEFAULT_STATISTICS,
    stat_mean_final_charge,
]

summary = summarize_results(
    results,
    statistics_functions=custom_statistics,
)
```
